# Exponential Smoothing

In the previous lecture, we decomposed nuclear generation, fitted a line to the estimated trend-cycle, and repeated the seasonal effects to forecast future generation. The forecast extended those estimates from one stopping point.

Today we will develop an alternative method for generating forecasts. While it preserves our intuition about time series data being some combination of trend, cycle, seasonality, and remainder, it estimates those components in an entirely different manner.

Decomposition-based forecasting estimates the trend-cycle and seasonal components separately, forecasts each component, and combines the results.

Exponential smoothing updates the components together in an iterative fashion. This method combines the observation with previous component estimates, using fitted parameters to control how much those estimates change. Forecasts are calculated from combinations of prior observations and estimated components, projected forward recursively.

In today's lecture we'll build that approach up, component by component, increasing model flexibility with each step, which may improve predictive accuracy.

## References

Adapted from Rob J. Hyndman and George Athanasopoulos, [Forecasting: Principles and Practice, Chapter 8](https://otexts.com/fpp3/exponential-smoothing.html), and their companion teaching materials. [The Pythonic edition](https://otexts.com/fpppy/08-exponential-smoothing.html) provides additional worked examples. Python implementation reference: [statsmodels exponential smoothing](https://www.statsmodels.org/stable/examples/notebooks/generated/exponential_smoothing.html).

Examples use historical Algerian exports, Australian population, and Australian holiday trips. [Data provenance](data/README.md) records their sources and units.

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt, ExponentialSmoothing

# Use bundled data locally, or the published course extract in Colab.
DATA_URL = (
    "https://raw.githubusercontent.com/olearydj/INSY7130/main/"
    "lectures/12b-exponential-smoothing/data/"
)

def load_lecture_data(filename):
    candidates = [
        Path("data") / filename,
        Path("content/lectures/12b-exponential-smoothing/data") / filename,
        Path("lectures/12b-exponential-smoothing/data") / filename,
    ]
    data_path = next((path for path in candidates if path.is_file()), None)
    if data_path is not None:
        return pd.read_csv(data_path)
    with urlopen(DATA_URL + filename, timeout=30) as response:
        return pd.read_csv(response)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False,
                     "axes.spines.right": False})

We'll first look at the Algerian Exports data, plotted below.

In [ ]:
exports = load_lecture_data("algerian-exports.csv").set_index("Year")["Exports"]

fig, ax = plt.subplots()
ax.plot(exports.index, exports, color="0.25", label="Observed exports")
ax.set(title="Algerian Exports, 1960–2017", xlabel="Year", ylabel="Exports (% of GDP)")
plt.show()

## Simple Exponential Smoothing

We will begin by exploring two simple ways to predict the next observation: repeat the latest value, or take the average of the historical values. We will use the Algerian exports series to see how a weighted average sits between those choices.

### Weighting Historical Observations

Suppose we have observations through time $T$ and want to predict what comes next. Two very simple methods, which can be used as baselines, are tabulated below.

| Forecast rule | Calculation | Treatment of history |
| --- | --- | --- |
| Naïve | $\hat{y}_{T+h\mid T}=y_T$ | All the weight goes to the latest observation |
| Historical average | $\hat{y}_{T+h\mid T}=\bar{y}=(y_1+\cdots+y_T)/T$ | Every observation receives equal weight |

Here, $h$ is the number of periods ahead, and $T$ is the last observed period. The equations follow [FPP Pythonic §5.2](https://otexts.com/fpppy/05-toolbox.html#some-simple-forecasting-methods).

These are two extremes. The naïve forecast is a random walk. Only the last observation matters. Historical average considers all prior observations, but gives them all equal weight.

Simple exponential smoothing (SES) lands somewhere in between. It gives recent observations more weight and gradually reduces the influence of older ones.

In [ ]:
baseline_years = np.arange(exports.index[-1] + 1, exports.index[-1] + 6)
naive_forecast = np.full(5, exports.iloc[-1])
mean_forecast = np.full(5, exports.mean())

fig, ax = plt.subplots()
ax.plot(exports.index, exports, color="0.3", label="Observed exports")
ax.plot(baseline_years, naive_forecast, "o--", label="Naïve forecast")
ax.plot(baseline_years, mean_forecast, "s--", label="Historical average forecast")
ax.axvline(exports.index[-1] + 0.5, color="0.6", linestyle=":", label="Forecast boundary")
ax.set(title="Baseline Forecasts From the 2017 Endpoint",
       xlabel="Year", ylabel="Exports (% of GDP)")
ax.legend()
plt.show()

### Exponential Weights

Let $\alpha$ be the weight on the latest observation. For $0<\alpha<1$, multiply by $1-\alpha$ each time we move one observation farther into the past:

$$
\alpha,\quad \alpha(1-\alpha),\quad \alpha(1-\alpha)^2,\quad\ldots
$$

That repeated multiplication gives exponential smoothing its name. The forecast is a weighted average of the observations, with older observations receiving less weight. The table and plot show how quickly those weights decrease for several choices of alpha. Age zero is the latest observation.

In [ ]:
age = np.arange(12)
weights = pd.DataFrame({f"alpha = {a:.1f}": a * (1-a)**age for a in [0.2, 0.4, 0.6, 0.8]},
                       index=pd.Index(age, name="Age (0 = newest)"))
display(weights.head(6).round(4))
fig, ax = plt.subplots()
for label in weights:
    ax.plot(age, weights[label], marker="o", label=label)
ax.set(xlabel="Age in observations (0 = newest)", ylabel="Weight",
       title="Larger Alpha Concentrates Weight on Recent Observations")
ax.legend()
plt.show()

With alpha 0.8, the newest three weights are 0.8, 0.16 and 0.032. With alpha 0.2, they are 0.2, 0.16 and 0.128. The larger alpha responds more strongly to the latest observation and forgets earlier observations faster.

Predict what will happen when alpha equals 1. Which baseline does it reduce to?

##### Answer

Alpha equal to one gives the latest observation all the weight. This recovers the naïve forecast.

### Update the Level

We can calculate the weighted average by keeping one running estimate, the level. Write $y_t$ for the observation at time $t$ and $\ell_{t-1}$ for the level estimated through time $t-1$. Update the level using

$$
\ell_t=\alpha y_t+(1-\alpha)\ell_{t-1}
$$

The next forecast uses that updated level:

$$
\hat y_{t+1\mid t}=\ell_t
$$

The notation $t+1\mid t$ means that we forecast time $t+1$ using information through time $t$. We can also write the update as a correction to the previous forecast:

$$
\ell_t=\ell_{t-1}+\alpha(y_t-\ell_{t-1})=\hat y_{t+1\mid t}
$$

Suppose the previous level was 30, the next observation is 40, and alpha is 0.2. The forecast error is 10. Add 20% of that error to the previous level: 30 + 0.2 × 10 = 32. The next forecast is 32.

In [ ]:
previous_level = 30.0
observation = 40.0
alpha = 0.2
new_level = previous_level + alpha * (observation - previous_level)
print(f"Previous forecast: {previous_level:.1f}")
print(f"Observation: {observation:.1f}")
print(f"Updated level and next forecast: {new_level:.1f}")

Now change alpha to 0.8 and run the cell again. Predict the result first, then explain why the update moves farther toward the observation.

##### Answer

The new level is 38: the previous level of 30 plus 80% of the error of 10. More of the latest surprise enters the next forecast.

### Forecast and Update Timing

To forecast an observation, use the level available before observing it:

$$
\hat y_{t\mid t-1}=\ell_{t-1}
$$

After observing $y_t$, calculate $\ell_t$ and use it for the next forecast. Start with an initial level, $\ell_0$. The pseudocode separates the forecast made beforehand from the level calculated afterward. Keep alpha fixed throughout these updates.

```text
INPUT:
    observations in time order
    smoothing weight alpha, between 0 and 1
    initial level, before the first observation

INITIALIZE:
    level = initial level

FOR each time t, in order:
    1. Forecast before seeing the observation
        prediction[t] = level

    2. Observe the actual value
        actual = observation[t]
        error = actual - prediction[t]

    3. Update the level
        level = level + alpha × error
        updated_level[t] = level

AFTER the last observation:
    prediction for the next period = level
```

Each prediction uses the level from the preceding update. The observation then supplies the error, and alpha determines how much of that error enters the new level.

The following cell uses matplotlib to animate this process.

It begins with 12 months of an illustrative monthly series. Alpha is fixed at 0.3, and the initial level before month 1 is 30. The blue shading marks the history already incorporated into the current level. Its left edge stays fixed as its right edge advances.

Press play or advance by frame to show each step for the following months:

- forecast the next month
- reveal the next observed value
- update the level

The orange square is the forecast made beforehand; the green diamond is the level after the update. Pause before revealing an observation and predict how the level will change.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# An illustrative series with a rise after the first twelve months.
timing_y = np.array([30., 32., 29., 31., 33., 30., 32., 31., 34., 32.,
                     33., 32., 38., 41., 39., 43., 40., 42., 39., 41.])
timing_alpha = 0.3
# Apply the same level updates to prepare the animation frames.
timing_levels = []
timing_current_level = 30.
for timing_observation in timing_y:
    timing_current_level += timing_alpha * (timing_observation - timing_current_level)
    timing_levels.append(timing_current_level)
timing_levels = np.array(timing_levels)
timing_months = np.arange(1, len(timing_y) + 1)

timing_fig, timing_ax = plt.subplots(figsize=(10, 5.8))
timing_fig.subplots_adjust(bottom=0.36, top=0.87)

def draw_timing(frame):
    # Each new month has three frames: forecast, observation, updated level.
    step, phase = divmod(frame, 3)
    known = 12 + step
    has_observation = known < len(timing_y)
    updated = has_observation and phase == 2
    incorporated = known + int(updated)
    visible = known + int(has_observation and phase >= 1)
    prior_level = timing_levels[known - 1]
    next_month = known + 1

    timing_ax.clear()
    timing_ax.axvspan(0.5, incorporated + 0.5, color="tab:blue", alpha=0.10)
    timing_ax.axvline(incorporated + 0.5, color="tab:blue", ls=":")
    timing_ax.plot(timing_months[:visible], timing_y[:visible], "o-",
                   color="0.35", label="Observed")
    timing_ax.plot(timing_months[:incorporated], timing_levels[:incorporated],
                   "D--", color="tab:green", ms=4, label="Updated level")
    timing_ax.plot([known, next_month], [prior_level, prior_level],
                   "--", color="tab:orange")
    timing_ax.plot(next_month, prior_level, "s", color="tab:orange", ms=9,
                   label="Forecast made beforehand", zorder=5)

    if has_observation and phase >= 1:
        actual = timing_y[known]
        error = actual - prior_level
        timing_ax.plot([next_month, next_month], [prior_level, actual],
                       ":", color="tab:red", lw=2, label="Forecast error")
    if updated:
        timing_ax.plot(next_month, timing_levels[known], "D", color="tab:green",
                       ms=9, zorder=6)

    if not has_observation or phase == 0:
        title = f"Forecast Month {next_month}"
        explanation = (
            f"History incorporated: months 1–{known}.  Alpha = {timing_alpha:.1f}\n"
            f"Forecast for month {next_month} = level after month {known} = {prior_level:.2f}\n"
            "The next observation has not been revealed."
        )
    elif phase == 1:
        title = f"Observe Month {next_month}"
        explanation = (
            f"Observation = {actual:.2f}; forecast made beforehand = {prior_level:.2f}\n"
            f"Forecast error = {actual:.2f} − {prior_level:.2f} = {error:+.2f}\n"
            "How far should the level move toward this observation?"
        )
    else:
        title = f"Update the Level Through Month {next_month}"
        explanation = (
            f"New level = {prior_level:.2f} + {timing_alpha:.1f} × ({actual:.2f} − {prior_level:.2f})"
            f" = {timing_levels[known]:.2f}\n"
            f"History incorporated: months 1–{next_month}. The left edge stays fixed.\n"
            f"This updated level becomes the forecast for month {next_month + 1}."
        )

    timing_ax.set(xlim=(0.5, len(timing_y) + 1.5), ylim=(27, 46),
                  xticks=np.arange(1, len(timing_y) + 2, 2),
                  xlabel="Month", ylabel="Illustrative monthly measurement", title=title)
    timing_ax.legend(loc="upper left", fontsize=8, ncol=2)
    timing_status.set_text(explanation)
    return []

timing_status = timing_fig.text(0.12, 0.07, "", fontsize=11, linespacing=1.7)
timing_fig.suptitle("Simple Exponential Smoothing: Forecast, Observe, Update", fontsize=14)
timing_animation = FuncAnimation(
    timing_fig, draw_timing, frames=3 * (len(timing_y) - 12) + 1,
    interval=1400, repeat=False,
)
timing_player = HTML(timing_animation.to_jshtml(default_mode="once"))
plt.close(timing_fig)  # Display only the player, rather than a duplicate static chart.
display(timing_player)

The forecast for each month stays at the previous level. Once the observation is revealed, the level moves 30% of the forecast error toward it. That updated level supplies the next month's forecast, retaining the earlier history through the recursive update.

Each update retains $1-\alpha$ of the previous level. Substitute the earlier updates into the current one to see how the weights on past observations arise:

$$
\begin{aligned}
\ell_t
&=\alpha y_t+(1-\alpha)\ell_{t-1}\\
&=\alpha y_t+\alpha(1-\alpha)y_{t-1}+(1-\alpha)^2\ell_{t-2}\\
&=\alpha y_t+\alpha(1-\alpha)y_{t-1}+\alpha(1-\alpha)^2y_{t-2}+(1-\alpha)^3\ell_{t-3}
\end{aligned}
$$

An observation enters with weight $\alpha$, and each subsequent update multiplies its weight by $1-\alpha$. The last term carries the remaining earlier history through the older level estimate. With alpha 0.3, an observation's weights become 0.30, 0.21, 0.147, and so on. The exponential decay arises from repeatedly applying the update rule.

Equivalently, after $T$ observations,

$$
\ell_T=\sum_{j=0}^{T-1}\alpha(1-\alpha)^j y_{T-j}+(1-\alpha)^T\ell_0
$$

The final term represents the remaining weight on the initial level.

The Python helper below implements the same calculation.

In [ ]:
def smooth(observations, alpha, initial_level):
    level = float(initial_level)
    forecasts, levels = [], []
    for observation in observations:
        forecasts.append(level)  # Available before this observation.
        level = alpha * observation + (1-alpha) * level
        levels.append(level)  # Available after this observation.
    return np.array(forecasts), np.array(levels)

### Alpha and Responsiveness

Compare three alpha values using the same initial level, 39.54. We will estimate that starting level shortly. Holding it fixed here lets us see the effect of alpha alone. Each dashed value is placed at the year it predicts.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True, sharey=True)
for ax, a in zip(axes, [0.2, 0.84, 1.0]):
    fitted, levels = smooth(exports.to_numpy(), a, 39.54)
    ax.plot(exports.index, exports, color="0.4", label="Observed")
    ax.plot(exports.index, fitted, color="tab:blue", linestyle="--", label="One-step forecast")
    ax.set(title=f"Alpha = {a}", ylabel="Exports (% of GDP)")
axes[0].legend(ncol=2)
axes[-1].set_xlabel("Year")
fig.tight_layout()
plt.show()

With alpha 0.2, one-step forecasts change slowly. With alpha 1, each forecast is exactly the previous observation, so the dashed series follows the observed series one year later. The intermediate setting responds quickly while retaining some earlier information.

Which line looks smoothest? Does that make it the best forecast? Explain why these are different questions.

##### Answer

Alpha 0.2 produces the smoothest sequence. A smoother sequence responds more slowly to a change. Forecast accuracy depends on how those responses compare with the observations that follow.

### Estimate Alpha and the Initial Level

We need to choose alpha and an initial level. As in regression, we choose values that minimize the sum of squared errors:

$$
\mathrm{SSE}=\sum_{t=1}^{T}(y_t-\hat y_{t\mid t-1})^2
$$

For each candidate pair, the software runs the updates through the series, calculates the fitted forecasts, and sums their squared errors. Numerical optimization searches for a pair with a smaller SSE.

`SimpleExpSmoothing` specifies the method. Setting `initialization_method="estimated"` allows the initial level to be estimated along with alpha. `.fit(optimized=True)` performs the search. `exports_values` contains the observed exports in chronological order, without the year index. `ses_model` specifies the model, while `ses_fit` contains the fitted parameters, fitted forecasts, and methods for generating future forecasts.

In [ ]:
exports_values = exports.to_numpy()
ses_model = SimpleExpSmoothing(exports_values, initialization_method="estimated")
ses_fit = ses_model.fit(optimized=True)
ses_alpha = float(ses_fit.params["smoothing_level"])
ses_initial_level = float(ses_fit.params["initial_level"])
print(f"Estimated alpha: {ses_alpha:.4f}")
print(f"Estimated initial level: {ses_initial_level:.2f}")
print(f"Sum of squared one-step errors: {ses_fit.sse:.2f}")

The fitted alpha is about 0.84 and the initial level is 39.54. The latest observation therefore has a large influence on the next forecast. Pass those same fitted parameters to our `smooth()` helper, then compare its one-step forecasts with `ses_fit.fittedvalues`, the library's one-step forecasts. Both forecast columns use the level before that row's observation; the final column shows the level after incorporating it. All four columns are in exports as a percentage of GDP.

In [ ]:
manual_forecasts, manual_updated_levels = smooth(
    exports_values, ses_alpha, ses_initial_level
)
library_forecasts = ses_fit.fittedvalues
ses_comparison = pd.DataFrame({
    "Observed exports": exports,
    "Manual forecast": manual_forecasts,
    "Library forecast": library_forecasts,
    "Level after observation": manual_updated_levels,
})
display(ses_comparison.head(6).round(2))
print("Manual and library forecasts agree across all years:",
      np.allclose(manual_forecasts, library_forecasts))

The manual and library forecast columns agree. In 1960, both use the initial level of 39.54. After observing exports of 39.04, the level updates to 39.12. That updated level becomes both methods' forecast for 1961. Each updated level therefore matches the following row's forecast, rather than the forecast in its own row.

These are in-sample predictions: alpha and the initial level were selected using this whole historical series. Although each update uses observations in order, its fitted parameters reflect the complete record. A forecast evaluation requires fitting on the available training history and comparing with later observations, as we did previously.

### Forecast Beyond the Last Observation

After the last observation, the method has one estimated level to carry forward:

$$
\hat y_{T+h\mid T}=\ell_T,\qquad h=1,2,\ldots
$$

Use `.forecast(5)` to produce five annual forecasts.

In [ ]:
future_years = np.arange(exports.index[-1]+1, exports.index[-1]+6)
forecast = pd.Series(ses_fit.forecast(5), index=future_years, name="Point forecast (% of GDP)")
display(forecast.round(2).to_frame())
fig, ax = plt.subplots()
ax.plot(exports.index, exports, color="0.3", label="Observed exports")
ax.plot(future_years, forecast, "o--", label="SES point forecast")
ax.axvline(exports.index[-1]+0.5, color="0.6", linestyle=":", label="Forecast boundary")
ax.set(xlabel="Year", ylabel="Exports (% of GDP)", title="Five Annual Forecasts From the 2017 Endpoint")
ax.legend()
plt.show()

The forecasts are all 22.44% of GDP. With alpha about 0.84, that is close to the last observed value of 22.64. The method has learned a current level.

Simple exponential smoothing has no slope or seasonal pattern to extend, so these point forecasts are flat.

To improve on this we need a better model, one that accounts for the missing components.

## Holt’s Trend Method

Holt’s method maintains a level and a slope. Both estimates can change when an observation arrives. The forecast at a particular stopping point extends the latest slope, so the line we project can change as we move through the history.

### Level and Slope Updates

Write $\ell_t$ for the level after observation $t$ and $b_t$ for the estimated increase per period.

The forecast follows the familiar straight-line equation $y = mx + b$: the intercept is the current level, the slope is the current estimated increase, and the horizontal distance is the number of periods ahead, $h$. In Holt’s notation, $b_t$ denotes the slope.

$$
\hat y_{t+h\mid t}=\ell_t+h b_t
$$

The level update combines the observation with the forecast made before it arrived:

$$
\ell_t=\alpha y_t+(1-\alpha)(\ell_{t-1}+b_{t-1})
$$

The slope update combines the newly estimated increase in level with the previous slope:

$$
b_t=\beta^*(\ell_t-\ell_{t-1})+(1-\beta^*)b_{t-1}
$$

Alpha controls how quickly the level responds. Beta-star controls how quickly the slope responds - it is a slope-smoothing control. Both weights lie between zero and one. A smaller beta-star gives the newly estimated change in level less weight. A larger beta-star allows the slope to change more quickly.

We need starting values for both level and slope. The fit estimates those two values and the two smoothing weights by minimizing squared fitted errors.

Use a previous level of 100 and a slope of 2 per period. The next forecast is 102. Suppose we observe 106. For this calculation, use alpha 0.5 and beta-star 0.25.

1. Update the level: 0.5 × 106 + 0.5 × 102 = 104.
2. Calculate the new increase in level: 104 − 100 = 4.
3. Update the slope: 0.25 × 4 + 0.75 × 2 = 2.5.
4. Forecast the next period: 104 + 2.5 = 106.5.

The code extends the same calculation to four forecast horizons.

In [ ]:
old_level, old_slope = 100.0, 2.0
y, alpha, beta = 106.0, 0.5, 0.25
prior_forecast = old_level + old_slope
new_level = alpha*y + (1-alpha)*prior_forecast
new_slope = beta*(new_level-old_level) + (1-beta)*old_slope
horizons = np.arange(1, 5)
display(pd.DataFrame({"Horizon": horizons,
                      "Forecast": new_level + horizons*new_slope}))

If beta-star is zero, what can still change, and what stays fixed?

##### Answer

The slope stays at its previous estimate. The level can still change through its own update, shifting the starting point of subsequent forecasts.

### Australian Population Forecasts

Australian population provides a clear example of a changing growth rate. Load the annual observations for 1960–2017 and express the population in millions of people.

In [ ]:
raw = load_lecture_data("australian-population.csv")
population = pd.Series(raw.Population.to_numpy()/1e6,
                       index=pd.PeriodIndex(raw.Year.astype(str), freq="Y"),
                       name="Population (millions)")
fig, ax = plt.subplots()
ax.plot(population.index.year, population, color="0.25")
ax.set(title="Australian Population", xlabel="Year", ylabel="Millions of people")
plt.show()

The population curve becomes steeper toward the end. Holt’s slope estimate can follow this increase in the growth rate.

`Holt` specifies the level-and-slope method. `initialization_method="estimated"` allows both starting values to be fitted. `.fit()` estimates those values ($\ell_0$ and $b_0$) and the smoothing weights ($\alpha$ and $\beta^*$). First inspect a fit to the whole record, then evaluate forecasts using a chronological split.

In [ ]:
holt = Holt(population, initialization_method="estimated").fit()
parameters = pd.Series({
    "Alpha": holt.params["smoothing_level"],
    "Beta-star": holt.params["smoothing_trend"],
    "Initial level (millions)": holt.params["initial_level"],
    "Initial slope (millions/year)": holt.params["initial_trend"],
})
display(parameters.round(4))
components = pd.DataFrame({"Observed": population,
                           "Forecast before observation": holt.fittedvalues,
                           "Updated level": holt.level,
                           "Updated slope": holt.trend,
                           "Fitted error": population-holt.fittedvalues})
display(components.head(6).round(4))

Alpha is close to one, so the updated level follows the observation closely. Beta-star is about 0.33, allowing the slope to change more gradually. The initial level is about 10.05 million and the initial slope about 0.224 million per year.

The table places the prior forecast beside the updated estimates. For each year, the forecast uses the preceding year’s level plus slope. The updated level and slope incorporate the observation in that row.

Plot the estimated level and slope through time, followed by the errors from the fitted forecasts.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
years = population.index.year
axes[0].plot(years, population, label="Observed", color="0.4")
axes[0].plot(years, holt.level, "--", label="Updated level")
axes[0].set_ylabel("Millions")
axes[0].legend()
axes[1].plot(years, holt.trend)
axes[1].set_ylabel("Millions/year")
axes[1].set_title("Updated Slope")
axes[2].plot(years, population-holt.fittedvalues)
axes[2].axhline(0, color="0.5", lw=1)
axes[2].set(ylabel="Error (millions)", xlabel="Year")
fig.suptitle("Holt’s Estimated Level, Slope, and Fitted Errors")
fig.tight_layout()
plt.show()

The slope is around 0.2 million people per year for much of the record, then rises toward 0.35 million near the end. The last panel shows the observation minus its fitted forecast. The change in slope explains why the final forecast differs from a straight line fitted across the entire history.

### Slope Smoothing

A beta-star of zero keeps the slope constant. As beta-star increases, the slope responds more strongly to the latest estimated change in level. The following calculation holds alpha and the starting values at their fitted values, then compares three beta-star settings. Watch how much the estimated slope moves between years.

In [ ]:
def holt_updates(observations, alpha, beta, initial_level, initial_slope):
    level, slope = initial_level, initial_slope
    slopes = []
    for observation in observations:
        prior_forecast = level + slope
        new_level = alpha*observation + (1-alpha)*prior_forecast
        slope = beta*(new_level-level) + (1-beta)*slope
        level = new_level
        slopes.append(slope)
    return np.array(slopes)

fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True, sharey=True)
for ax, beta in zip(axes, [0.0, 0.3, 0.8]):
    slopes = holt_updates(population, holt.params["smoothing_level"], beta,
                          holt.params["initial_level"], holt.params["initial_trend"])
    ax.plot(population.index.year, slopes)
    ax.set(title=f"Beta-star = {beta}", ylabel="Millions/year")
axes[-1].set_xlabel("Year")
fig.suptitle("Slope Updates With Different Beta-Star Values")
fig.tight_layout()
plt.show()

At zero, the estimated slope stays at its starting value. At 0.3 it follows the changing growth rate more gradually than at 0.8. The larger value responds quickly but also follows more of the year-to-year variation. The fit chooses beta-star by comparing the resulting forecast errors.

### Linear Trend Forecasts

`.forecast(10)` projects the last level and slope ten years ahead. The table separates the starting level from the accumulated growth at each horizon.

In [ ]:
horizon = np.arange(1, 11)
last_level, last_slope = holt.level.iloc[-1], holt.trend.iloc[-1]
future = holt.forecast(10)
display(pd.DataFrame({"Year": future.index.year, "Horizon": horizon,
                      "Last level": last_level,
                      "Added growth": horizon*last_slope,
                      "Forecast": future.to_numpy()}).round(3))
assert np.allclose(future, last_level+horizon*last_slope)
fig, ax = plt.subplots()
ax.plot(population.index.year, population, color="0.3", label="Observed")
ax.plot(future.index.year, future, "o--", label="Holt forecast")
ax.axvline(2017.5, color="0.6", ls=":")
ax.set(title="Ten-Year Forecast From the 2017 Endpoint", xlabel="Year", ylabel="Millions of people")
ax.legend()
plt.show()

Does the slope keep learning during these ten forecast years?

##### Answer

This forecast uses the final slope, about 0.369 million people per year, at all ten horizons. When a new observation arrives, we can update the level and slope and issue another forecast.

### Damped Trends

Holt’s forecast extends the same slope indefinitely. For a long horizon, that may project more growth or decline than we expect to continue. A damped trend reduces the slope’s contribution as the horizon increases. Phi controls the reduction. Values closer to one retain the slope for longer.

The damped forecast replaces $h$ with the sum $\phi+\phi^2+\cdots+\phi^h$, reducing the contribution of each additional period’s growth.

$$
\hat y_{t+h\mid t}=\ell_t+(\phi+\phi^2+\cdots+\phi^h)b_t
$$

The growth contributions for the first three steps are $\phi b_t$, $\phi^2 b_t$, and $\phi^3 b_t$. For $0<\phi<1$, each is smaller than the previous one. At phi one, their sum is $h b_t$, which gives Holt’s linear forecast.

Damping also enters the updates:

$$
\ell_t=\alpha y_t+(1-\alpha)(\ell_{t-1}+\phi b_{t-1})
$$

$$
b_t=\beta^*(\ell_t-\ell_{t-1})+(1-\beta^*)\phi b_{t-1}
$$

The forecast can follow a trend over the next few periods while gradually flattening over a longer horizon.

Hold the final level and slope fixed and compare phi values of 1, 0.98, and 0.9. A thirty-year horizon makes the difference in forecast shape visible.

In [ ]:
horizon = np.arange(1, 31)
fig, ax = plt.subplots()
for phi in [1.0, 0.98, 0.9]:
    growth = np.cumsum(phi**horizon)*last_slope  # Add the damped increments through each horizon.
    ax.plot(2017+horizon, last_level+growth, label=f"Phi = {phi}")
ax.scatter([2017], [last_level], color="black", label="Starting level")
ax.set(title="Same Starting Level and Slope, Different Damping",
       xlabel="Year", ylabel="Millions of people")
ax.legend()
plt.show()

A smaller phi makes the forecast flatten sooner. This comparison changed only the forecast rule. We will now estimate the complete damped method and assess its forecasts alongside SES and Holt.

### Compare Forecasts on Withheld Years

Train through 2010 and withhold 2011–2017. Fit each method on the training observations and forecast the same seven years.

`ExponentialSmoothing` provides a general interface to a broad range of exponential smoothing methods, including combinations of trend, damping, and seasonality beyond those used in this lecture. Its arguments specify which components to include and whether to damp the trend. Here we use it for all three nonseasonal methods:

| Method | `ExponentialSmoothing` settings |
| --- | --- |
| SES | `trend=None, seasonal=None` |
| Holt’s linear trend | `trend="add", seasonal=None` |
| Damped Holt | `trend="add", damped_trend=True` |

Damping is off by default, and `seasonal=None` excludes seasonality in all three models. Each fit estimates its smoothing weights and starting values. For damped Holt, we also estimate phi within 0.8–0.98, keeping the forecast close to linear over short horizons while allowing it to flatten over longer ones.

In [ ]:
training = population[population.index.year <= 2010]
test = population[population.index.year > 2010]
models = {
    "SES": ExponentialSmoothing(
        training, trend=None, seasonal=None, initialization_method="estimated").fit(),
    "Holt": ExponentialSmoothing(
        training, trend="add", seasonal=None, initialization_method="estimated").fit(),
    "Damped Holt": ExponentialSmoothing(
        training, trend="add", damped_trend=True, seasonal=None,
        initialization_method="estimated",
        bounds={"damping_trend": (0.8, 0.98)}).fit(),
}
predictions = pd.DataFrame({name: model.forecast(len(test)) for name, model in models.items()})
errors = predictions.rsub(test, axis="index")  # Actual minus predicted, aligned by year.
scores = pd.DataFrame({"MAE (millions)": errors.abs().mean(),
                       "RMSE (millions)": np.sqrt(errors.pow(2).mean())})
display(scores.round(3))
fig, ax = plt.subplots()
ax.plot(population.index.year, population, color="0.3", label="Observed")
for name in predictions:
    ax.plot(predictions.index.year, predictions[name], "o--", label=name)
ax.axvline(2010.5, color="0.6", ls=":", label="Forecast boundary")
ax.set(ylim=(population[population.index.year >= 2000].min()-0.3, population.max()+0.3),
       xlim=(2000, 2018), title="Forecasts Made at the End of 2010",
       xlabel="Year", ylabel="Millions of people")
ax.legend()
plt.show()

The held-out RMSE is about 1.63 million people for SES, 0.15 million for Holt, and 0.20 million for damped Holt. SES holds population constant while the observed population keeps growing. Holt captures that growth best on these seven years. Damping reduces the projected growth and leaves slightly larger errors.

What has Holt added to SES, and what important pattern is still absent?

##### Answer

Holt maintains a changing slope as well as a level. A recurring seasonal pattern needs another set of estimates, one for each position in the cycle.

## Holt–Winters Seasonal Methods

Holt–Winters adds seasonality to the level and slope.

### Additive Holt–Winters

Additive seasonality describes seasonal differences in the original measurement units. Before updating the level, subtract the previous estimate for the same season. Update the slope as in Holt’s method. Then revise the seasonal estimate using the difference between the observation and the level-and-slope prediction.

Let $m$ be the number of observations in a seasonal cycle. The estimate $s_{t-m}$ is the effect for this season from one cycle earlier. For quarterly data, $m=4$.

**Level:** seasonally adjust the observation, then combine it with the previous level-and-slope forecast.

$$
\ell_t=\alpha(y_t-s_{t-m})+(1-\alpha)(\ell_{t-1}+b_{t-1})
$$

**Slope:** combine the new change in level with the previous slope.

$$
b_t=\beta^*(\ell_t-\ell_{t-1})+(1-\beta^*)b_{t-1}
$$

**Seasonal effect:** compare the observation with the level-and-slope prediction, then combine that difference with the old effect for this season.

$$
s_t=\gamma(y_t-\ell_{t-1}-b_{t-1})+(1-\gamma)s_{t-m}
$$

Alternatively, the seasonal update can be expressed using the newly updated level:

$$
s_t=\gamma^*(y_t-\ell_t)+(1-\gamma^*)s_{t-m}
$$

The forms are equivalent when $\gamma=\gamma^*(1-\alpha)$; their smoothing coefficients therefore have different numerical values.

The new smoothing weight, gamma, controls how quickly the seasonal effect changes. The forecast uses the latest level and slope plus the effect for the matching season:

$$
\hat y_{t+h\mid t}=\ell_t+h b_t+s_{\text{matching season}}
$$

We now need initial estimates for the level, slope, and each seasonal position. Quarterly data require four initial seasonal estimates.

Suppose the previous level is 100, the slope is 2, and the effect for this quarter is 10. The forecast made before the observation is 100 + 2 + 10 = 112. We observe 120. Use alpha 0.5, beta-star 0.25, and gamma 0.2.

1. Seasonally adjust the observation: 120 − 10 = 110.
2. Update the level: 0.5 × 110 + 0.5 × 102 = 106.
3. Update the slope: 0.25 × (106 − 100) + 0.75 × 2 = 3.
4. Calculate the new seasonal signal: 120 − 102 = 18.
5. Update this quarter’s seasonal effect: 0.2 × 18 + 0.8 × 10 = 11.6.

In [ ]:
old_level, old_slope, old_season = 100., 2., 10.
y, alpha, beta, gamma = 120., 0.5, 0.25, 0.2
prior_base = old_level + old_slope
level = alpha*(y-old_season) + (1-alpha)*prior_base
slope = beta*(level-old_level) + (1-beta)*old_slope
season = gamma*(y-prior_base) + (1-gamma)*old_season
display(pd.Series({"Forecast before observation": prior_base+old_season,
                   "Observed": y, "Updated level": level,
                   "Updated slope": slope, "Updated seasonal effect": season}))

Should next quarter’s forecast use the newly updated seasonal effect of 11.6?

##### Answer

Next quarter uses its own seasonal estimate. The effect of 11.6 replaces the estimate for the quarter just observed and is used when that quarter comes around again.

### Seasonal Smoothing

With gamma zero, the seasonal estimate stays fixed. A larger gamma makes it respond more strongly to the latest observation for that season. The table holds the old effect of 10 and the new seasonal signal of 18 fixed while varying the update weight.

In [ ]:
gammas = np.array([0., 0.1, 0.2, 0.5])
new_signal = y-prior_base  # Seasonal signal in the worked observation.
display(pd.DataFrame({"Gamma": gammas, "Old effect": old_season,
                      "New seasonal signal": new_signal,
                      "Updated effect": gammas*new_signal+(1-gammas)*old_season}))

### Multiplicative Holt–Winters

When seasonal swings grow with the level, express the seasonal effect as a multiplier. An additive effect of 20 means 20 units above the baseline. A factor of 1.2 means 20% above it. At a baseline of 100 that gives 120, and at a baseline of 200 it gives 240.

Multiplicative Holt–Winters uses division to remove seasonality and multiplication to restore it.

$$
\begin{aligned}
\hat y_{t+h\mid t}&=(\ell_t+h b_t)s_{\text{matching season}}\\
\ell_t&=\alpha\frac{y_t}{s_{t-m}}+(1-\alpha)(\ell_{t-1}+b_{t-1})\\
b_t&=\beta^*(\ell_t-\ell_{t-1})+(1-\beta^*)b_{t-1}\\
s_t&=\gamma\frac{y_t}{\ell_{t-1}+b_{t-1}}+(1-\gamma)s_{t-m}
\end{aligned}
$$

The level update uses the seasonally adjusted observation $y_t/s_{t-m}$. The seasonal update uses the ratio of the observation to the previous level-and-slope prediction. The slope update is the same as in the additive method. This implementation requires positive observations.

Compare an additive effect and a multiplicative factor at two baseline levels.

In [ ]:
bases = np.array([100., 200.])
display(pd.DataFrame({"Projected level": bases,
                      "Additive effect +20": bases+20,
                      "Multiplicative factor 1.2": bases*1.2}))

Additive effects describe differences in the original units. Multiplicative factors describe proportions of the underlying level. When centered in the usual way, four quarterly additive effects sum to zero and four quarterly multiplicative factors average one.

### Australian Holiday Trips

Australian domestic overnight holiday trips have a strong quarterly pattern. The series below sums holiday trips across Australian regions for 1998–2017. The original measurements are in thousands of trips. Divide by 1,000 to express millions, then inspect how the seasonal swings change as the level rises.

In [ ]:
raw = load_lecture_data("australian-holiday-trips.csv")
trips = pd.Series(raw.Trips.to_numpy()/1000,
                  index=pd.PeriodIndex(raw.Quarter.str.replace(" ", ""), freq="Q"),
                  name="Holiday trips (millions)")
fig, ax = plt.subplots()
ax.plot(trips.index.to_timestamp(), trips, color="0.25")
ax.set(title="Australian Domestic Overnight Holiday Trips", xlabel="Year", ylabel="Millions of trips")
plt.show()

### Fit Both Seasonal Forms

Fit both seasonal forms with `ExponentialSmoothing`. Both have an additive trend and four observations per seasonal cycle. `seasonal="add"` estimates seasonal differences, while `seasonal="mul"` estimates seasonal factors. `.fit()` estimates the smoothing weights and starting values by minimizing squared fitted errors.

In [ ]:
fits = {
    "Additive": ExponentialSmoothing(trips, trend="add", seasonal="add",
        seasonal_periods=4, initialization_method="estimated").fit(),
    "Multiplicative": ExponentialSmoothing(trips, trend="add", seasonal="mul",
        seasonal_periods=4, initialization_method="estimated").fit(),
}
parameter_table = pd.DataFrame({name: {
    "Alpha": fit.params["smoothing_level"],
    "Beta-star": fit.params["smoothing_trend"],
    "Gamma": fit.params["smoothing_seasonal"],
    "Initial level (millions)": fit.params["initial_level"],
    "Initial slope (millions/quarter)": fit.params["initial_trend"],
} for name, fit in fits.items()})
display(parameter_table.round(4))

Both fits put gamma at or very near zero, keeping the estimated seasonal pattern almost fixed. The plots below show how substantial that repeated pattern is. Level and slope continue to update through alpha and beta-star.

### Inspect the Components

Plot the estimated level, slope, seasonal component, and fitted errors. Read the seasonal axes carefully: additive seasonality is in millions of trips, while multiplicative seasonality is a factor. These are the raw library estimates. Their centering can differ slightly from the conventional zero or one because level and seasonal scaling are fitted together.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(11, 9), sharex=True)
for col, (name, fit) in enumerate(fits.items()):
    values = [fit.level, fit.trend, fit.season, trips-fit.fittedvalues]
    labels = ["Level (millions)", "Slope (millions/qtr)",
              "Season (millions)" if name == "Additive" else "Season (factor)",
              "Error (millions)"]
    for row, (value, label) in enumerate(zip(values, labels)):
        axes[row, col].plot(trips.index.to_timestamp(), value)
        axes[row, col].set_ylabel(label)
    axes[0, col].set_title(name)
    axes[-1, col].set_xlabel("Year")
fig.suptitle("Holt–Winters Components and Fitted Errors")
fig.tight_layout()
plt.show()
add = fits["Additive"]
display(pd.DataFrame({"Observed": trips, "Forecast before observation": add.fittedvalues,
                      "Updated level": add.level, "Updated slope": add.trend,
                      "Updated season": add.season}).head(8).round(3))

The seasonal estimates repeat each year, while the level and slope follow slower changes. The table shows the same timing as before: each fitted forecast uses previous estimates, and the updated estimates incorporate that row’s observation. An updated seasonal estimate belongs to the quarter in that row.

### Seasonal Forecast Construction

At the end of 2017 Q4, project the level and slope for eight quarters. Add the last four seasonal estimates in quarter order, repeating them for the second year. The table makes each part of this additive forecast visible and the code checks it against `.forecast(8)`.

In [ ]:
horizon = np.arange(1, 9)
forecasts = pd.DataFrame({name: fit.forecast(8) for name, fit in fits.items()})
base = add.level.iloc[-1] + horizon*add.trend.iloc[-1]
season = np.tile(add.season.iloc[-4:].to_numpy(), 2)  # Repeat the final annual cycle.
display(pd.DataFrame({"Quarter": forecasts.index.astype(str),
                      "Projected level": base, "Matching seasonal effect": season,
                      "Forecast": base+season}).round(3))
assert np.allclose(base+season, forecasts["Additive"])
fig, ax = plt.subplots()
ax.plot(trips.index.to_timestamp(), trips, color="0.3", label="Observed")
for name in forecasts:
    ax.plot(forecasts.index.to_timestamp(), forecasts[name], "o--", label=name)
ax.axvline(pd.Timestamp("2018-01-01"), color="0.6", ls=":")
ax.set(xlim=(pd.Timestamp("2010-01-01"), pd.Timestamp("2020-01-01")),
       title="Two-Year Forecasts From the End of 2017", xlabel="Year", ylabel="Millions of trips")
ax.legend()
plt.show()

Why do the multiplicative forecasts develop larger seasonal swings as their level rises?

##### Answer

The same seasonal factor produces a larger absolute swing as the projected level rises. The additive method repeats effects in the original units. To compare forecasting accuracy, we would fit both methods before a common holdout period and compare their errors on those future quarters.

### Damped Seasonal Forecasts

Damping combines with seasonal forecasting in the same way it did with Holt’s method. Reduce the future growth contribution, then apply the matching seasonal factor. The next fit uses damped trend and multiplicative seasonality on the holiday-trips series. A longer horizon makes the effect of damping easier to see.

In [ ]:
damped = ExponentialSmoothing(trips, trend="add", damped_trend=True,
    seasonal="mul", seasonal_periods=4, initialization_method="estimated",
    bounds={"damping_trend": (0.8, 0.98)}).fit()
long_forecasts = pd.DataFrame({
    "Undamped multiplicative": fits["Multiplicative"].forecast(40),
    "Damped multiplicative": damped.forecast(40),
})
print(f"Estimated phi: {damped.params['damping_trend']:.3f}")
fig, ax = plt.subplots()
ax.plot(trips.iloc[-12:].index.to_timestamp(), trips.iloc[-12:], color="0.3", label="Observed")
for name in long_forecasts:
    ax.plot(long_forecasts.index.to_timestamp(), long_forecasts[name], label=name)
ax.set(title="Damped and Undamped Seasonal Forecast Shapes", xlabel="Year", ylabel="Millions of trips")
ax.legend()
plt.show()

## Exponential Smoothing Summary

The equations below recap how the methods build on one another. The component estimates are level $\ell_t$, slope $b_t$, and seasonal effect or factor $s_t$. The smoothing weights are $\alpha$, $\beta^*$, and $\gamma$. For damped forecasts, define $\phi_h=\phi+\phi^2+\cdots+\phi^h$.

These comparisons follow the organization of [FPP Pythonic §8.4, Table 8.7](https://otexts.com/fpppy/08-exponential-smoothing.html#a-taxonomy-of-exponential-smoothing-methods), using the seasonal-update convention introduced in this lecture.

### Nonseasonal Formulations

Read across each row to see how adding a slope and then damping changes the calculation. A dash means the method has no slope component.

| Equation | SES | Holt | Damped Holt |
| --- | --- | --- | --- |
| Forecast | $\hat y_{t+h\mid t}=\ell_t$ | $\hat y_{t+h\mid t}=\ell_t+h b_t$ | $\hat y_{t+h\mid t}=\ell_t+\phi_h b_t$ |
| Level update | $\ell_t=\alpha y_t+(1-\alpha)\ell_{t-1}$ | $\ell_t=\alpha y_t+(1-\alpha)(\ell_{t-1}+b_{t-1})$ | $\ell_t=\alpha y_t+(1-\alpha)(\ell_{t-1}+\phi b_{t-1})$ |
| Slope update | — | $b_t=\beta^*(\ell_t-\ell_{t-1})+(1-\beta^*)b_{t-1}$ | $b_t=\beta^*(\ell_t-\ell_{t-1})+(1-\beta^*)\phi b_{t-1}$ |

### Seasonal Formulations

Both methods below extend Holt with a seasonal component. Let $m$ be the number of observations per cycle and $S_h$ be the latest available seasonal estimate for the season being forecast. Additive effects are in the original measurement units; multiplicative effects are factors.

| Equation | Additive Holt–Winters | Multiplicative Holt–Winters |
| --- | --- | --- |
| Forecast | $\hat y_{t+h\mid t}=\ell_t+h b_t+S_h$ | $\hat y_{t+h\mid t}=(\ell_t+h b_t)S_h$ |
| Level update | $\ell_t=\alpha(y_t-s_{t-m})+(1-\alpha)(\ell_{t-1}+b_{t-1})$ | $\ell_t=\alpha\frac{y_t}{s_{t-m}}+(1-\alpha)(\ell_{t-1}+b_{t-1})$ |
| Slope update | $b_t=\beta^*(\ell_t-\ell_{t-1})+(1-\beta^*)b_{t-1}$ | $b_t=\beta^*(\ell_t-\ell_{t-1})+(1-\beta^*)b_{t-1}$ |
| Seasonal update | $s_t=\gamma(y_t-\ell_{t-1}-b_{t-1})+(1-\gamma)s_{t-m}$ | $s_t=\gamma\frac{y_t}{\ell_{t-1}+b_{t-1}}+(1-\gamma)s_{t-m}$ |

For a damped seasonal method, replace future growth $h b_t$ with $\phi_h b_t$ in the forecast. In the update equations, replace each previous slope $b_{t-1}$ with $\phi b_{t-1}$. The same substitutions apply to additive and multiplicative seasonality.

### ExponentialSmoothing Options

All rows use `ExponentialSmoothing(data, ..., initialization_method="estimated").fit()`. Trend and seasonality default to `None`, and damping defaults to `False`. For seasonal models, `m` is the number of observations in the repeating cycle.

| Model | Trend | Seasonality | Forecast pattern | Options |
| --- | --- | --- | --- | --- |
| SES | None | None | Flat at the final level | `trend=None, seasonal=None` |
| Holt | Linear | None | Extends the final slope | `trend="add", seasonal=None` |
| Damped Holt | Damped | None | Progressively flattens | `trend="add", damped_trend=True` |
| Additive Holt–Winters | Linear | Additive | Fixed seasonal differences around a projected trend | `trend="add", seasonal="add", seasonal_periods=m` |
| Multiplicative Holt–Winters | Linear | Multiplicative | Seasonal swings scale with the projected level | `trend="add", seasonal="mul", seasonal_periods=m` |
| Damped seasonal model | Damped | Additive or multiplicative | Trend progressively flattens while seasonality repeats | Add `damped_trend=True` to either seasonal model |

- **You choose the structure:** trend, seasonality, damping, and seasonal period.
- **Fitting estimates the parameters:** smoothing weights, initial components, and phi when damping is included.
- **Forecast evaluation compares the choices:** more components do not guarantee better forecasts.

Parameter roles: alpha controls level smoothing, beta-star controls slope smoothing, gamma controls seasonal smoothing, and phi controls damping.

### A Starting Model for Seasonal Forecasting

Damped Holt–Winters with multiplicative seasonality combines the ideas developed in this lecture: level and slope adapt to observations, damping moderates continued growth, and seasonal factors scale with the level. [FPP Pythonic §8.3](https://otexts.com/fpppy/08-exponential-smoothing.html#holt-winters-damped-method) identifies this combination as often producing accurate, robust seasonal forecasts. We fitted it in Damped Seasonal Forecasts; its general specification is:

```python
ExponentialSmoothing(
    data,
    trend="add",
    damped_trend=True,
    seasonal="mul",
    seasonal_periods=m,
    initialization_method="estimated",
).fit()
```

Here, `data` is the observed series and `m` is the number of observations per seasonal cycle. Multiplicative seasonality requires strictly positive observations.

Damped Holt–Winters with multiplicative seasonality often works well for seasonal data and is a useful starting model. Compare its forecasts with simpler alternatives on withheld observations.